# Job Hunt Agent Multi System

### Dependencies

In [1]:
%pip install openai beautifulsoup4 requests reportlab pdfminer.six python-docx gradio


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


### Imports

In [2]:
import os, re, json, requests, pathlib, tempfile, uuid
from typing import List, Dict, Any
from dataclasses import dataclass
from bs4 import BeautifulSoup
from urllib.parse import urlparse

from docx import Document
from pdfminer.high_level import extract_text as pdf_extract_text
from reportlab.lib.pagesizes import LETTER
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.units import inch
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.enums import TA_JUSTIFY, TA_LEFT

from openai import OpenAI
import gradio as gr

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Client setup

In [ ]:
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY") or "sk-REPLACE-WITH-YOUR-KEY"
if OPENAI_API_KEY.startswith("sk-REPLACE"):
    print("Warning: OPENAI_API_KEY is not set.")
os.environ["OPENAI_API_KEY"] = "sk-REPLACE-WITH-YOUR-KEY"
print("API key set ✔️")

API key set ✔️


In [4]:
client = OpenAI()
MODEL  = "gpt-4o-mini"
print("OpenAI client ready ✔️")

OpenAI client ready ✔️


### Helper functions

In [5]:
def extract_text_from_pdf(pdf_path: str) -> str:
    text = pdf_extract_text(pdf_path) or ""
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text[:200_000]


def sniff_contact(cv_text: str) -> Dict[str, str]:
    email = re.search(r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}", cv_text)
    phone = re.search(r"(\+?\d[\d \-()]{7,})", cv_text)
    first_lines = [ln.strip() for ln in cv_text.splitlines()[:10] if ln.strip()]
    name = ""
    for ln in first_lines:
        if (email and email.group(0) in ln) or (phone and phone.group(0) in ln):
            continue
        if len(ln.split()) <= 6:
            name = ln
            break
    return {
        "name":     name or "Candidate",
        "email":    email.group(0) if email else "",
        "phone":    phone.group(0) if phone else "",
        "location": "",
    }

### Export helpers

In [6]:
def save_text_as_docx(text: str, file_path: str) -> str:
    doc = Document()
    for p_text in re.split(r"\n\s*\n", text):
        if p_text.strip():
            doc.add_paragraph(p_text.strip())
    doc.save(file_path)
    return file_path


def save_text_as_pdf(text: str, file_path: str) -> str:
    text = text or ""
    doc  = SimpleDocTemplate(file_path, pagesize=LETTER)
    styles = getSampleStyleSheet()
    flow = []
    for p_text in re.split(r"\n\s*\n", text):
        if p_text.strip():
            flow.append(Paragraph(str(p_text).replace("\n", "<br/>"), styles["Normal"]))
    doc.build(flow)
    return file_path


def save_cover_letter_pdf(letter_text: str, file_path: str) -> str:
    os.makedirs(os.path.dirname(file_path) or ".", exist_ok=True)
    doc = SimpleDocTemplate(file_path, pagesize=LETTER,
                            rightMargin=72, leftMargin=72,
                            topMargin=72, bottomMargin=72)
    styles = getSampleStyleSheet()
    base   = ParagraphStyle("Body",   parent=styles["Normal"],
                            fontName="Times-Roman", fontSize=11,
                            leading=15, alignment=TA_JUSTIFY)
    header = ParagraphStyle("Header", parent=styles["Normal"],
                            fontName="Times-Bold",  fontSize=14,
                            leading=18, alignment=TA_LEFT, spaceAfter=12)
    chunks = letter_text.strip().splitlines()
    header_lines, body_lines, hit_blank = [], [], False
    for line in chunks:
        if not hit_blank and line.strip() == "":
            hit_blank = True; continue
        (header_lines if not hit_blank else body_lines).append(line)
    flow = []
    if header_lines:
        flow.append(Paragraph("<br/>".join(e for e in header_lines if e.strip()), header))
        flow.append(Spacer(1, 0.2 * inch))
    body = "\n".join(body_lines) if body_lines else letter_text
    for p in [p.strip() for p in re.split(r"\n\s*\n", body) if p.strip()]:
        flow.append(Paragraph(p.replace("\n", "<br/>"), base))
        flow.append(Spacer(1, 0.18 * inch))
    doc.build(flow)
    return file_path

def safe_filename(text: str) -> str:
    return re.sub(r'[^a-zA-Z0-9_-]+', '_', text).strip('_')

### Company name resolver
*(must be defined before CoverLetterAgent)*

In [7]:
# Job-board domains that should never become the company name
_JOB_BOARD_DOMAINS = {
    "linkedin", "indeed", "glassdoor", "monster", "ziprecruiter",
    "lever", "greenhouse", "workday", "icims", "taleo", "smartrecruiters",
    "jobs", "careers", "recruitee", "ashby", "bamboohr", "jobvite",
    "welcometothejungle", "otta", "remotive", "angellist", "wellfound",
}

def resolve_company_name(job: dict) -> str:
    """
    Return the best-guess company name from a scraped job dict.
    Falls back through: company_name field → title parsing → description → domain.
    Never returns a job-board domain name.
    """
    def _clean(s: str) -> str:
        return re.sub(r"[^A-Za-z0-9 &.,'()-]+", " ", s or "").strip()

    # 1. Use scraped company_name if it's not a job-board name
    scraped = _clean(job.get("company_name") or "")
    if scraped and scraped.lower() not in _JOB_BOARD_DOMAINS:
        return scraped

    # 2. Parse from title e.g. "Software Engineer at DotControl | LinkedIn"
    title = _clean(job.get("title_raw") or "")
    for sep in (" at ", " @ "):
        if sep in title:
            candidate = title.split(sep)[-1]
            candidate = re.split(r"[|·\-\u2013\u2014]", candidate)[0].strip()
            if candidate and candidate.lower() not in _JOB_BOARD_DOMAINS:
                return candidate

    # 3. Parse from description: "Company: Acme Corp"
    desc = (job.get("description") or "")[:2000]
    m = re.search(
        r"(?:company|employer|organisation|organization)\s*[:\-]\s*([A-Za-z0-9&.,'\- ]{2,60})",
        desc, re.I
    )
    if m:
        candidate = m.group(1).strip().rstrip(".,")
        if candidate and candidate.lower() not in _JOB_BOARD_DOMAINS:
            return candidate

    # 4. Domain fallback — skip known job boards
    host = urlparse(job.get("url") or "").netloc.lower()
    host = host.removeprefix("www.").split(":")[0]
    core = host.split(".")[-2] if "." in host else host
    if core and core not in _JOB_BOARD_DOMAINS:
        return core.replace("-", " ").title()

    return "Company"  # last resort

### Job scraper

In [8]:
class RoleScraper:
    UA = "Mozilla/5.0 (JobAgents/1.0)"

    @staticmethod
    def scrape(url: str, timeout: int = 12) -> Dict[str, str]:
        dom = (urlparse(url).netloc or "").lower()
        try:
            html = requests.get(url, timeout=timeout,
                                headers={"User-Agent": RoleScraper.UA}).text
        except Exception:
            return {"url": url, "title_raw": "", "description": "",
                    "company_name": "", "gated": True}

        try:
            soup = BeautifulSoup(html, "lxml")
        except Exception:
            soup = BeautifulSoup(html, "html.parser")

        ogt   = soup.select_one('meta[property="og:title"], meta[name="og:title"]')
        title = ((ogt.get("content") or "").strip() if ogt and ogt.get("content")
                 else (soup.title.get_text(strip=True) if soup.title else ""))[:300]

        desc, company = "", ""
        for tag in soup.select('script[type="application/ld+json"]'):
            try:
                data = json.loads(tag.string or "")
            except Exception:
                continue
            for obj in (data if isinstance(data, list) else [data]):
                if isinstance(obj, dict) and "JobPosting" in str(obj.get("@type", "")):
                    if isinstance(obj.get("description"), str) and len(obj["description"]) > len(desc):
                        desc = obj["description"]
                    org = obj.get("hiringOrganization") or {}
                    nm  = (org.get("name") if isinstance(org, dict) else org) or ""
                    if isinstance(nm, str) and len(nm) > len(company):
                        company = nm.strip()

        if desc:
            desc = re.sub(r"<br\s*/?>", "\n", desc, flags=re.I)
            desc = re.sub(r"<[^>]+>", "", desc)

        if len(desc) < 200:
            selectors = [
                ".opening .content", ".opening .description",
                ".posting .section", ".posting .content",
                "article#job-application", ".job-body", ".job__description",
                "[data-ashby-job-posting-description]",
                "section", "article", "div",
            ]
            blocks, seen = [], set()
            for sel in selectors:
                for node in soup.select(sel):
                    txt = node.get_text("\n", strip=True)
                    if txt and len(txt) > 200:
                        h = hash(txt)
                        if h not in seen:
                            seen.add(h); blocks.append(txt)
            desc = max(blocks, key=len) if blocks else soup.get_text("\n", strip=True)

        desc = re.sub(r"[ \t]+\n", "\n", desc or "")
        desc = re.sub(r"\n{3,}", "\n\n", desc).strip()[:20_000]

        # FIX: return statement was accidentally removed — restored here
        return {"url": url, "title_raw": title, "description": desc,
                "company_name": company[:200] if company else "",
                "gated": len(desc) < 200}

### Base agent

In [9]:
@dataclass
class AgentMessage:
    role:    str
    content: str

@dataclass
class BaseAgent:
    name:          str
    system_prompt: str

    def call_openai(self, messages: List[AgentMessage], max_tokens: int = 2000) -> str:
        payload = [{"role": m.role, "content": m.content} for m in messages]
        payload.insert(0, {"role": "system", "content": self.system_prompt})
        completion = client.chat.completions.create(
            model=MODEL,
            max_tokens=max_tokens,
            temperature=0.45,
            messages=payload,
        )
        return completion.choices[0].message.content.strip()

### Cover letter agent

In [10]:
class CoverLetterAgent(BaseAgent):

    # Strict formula template (from main.ipynb)
    _STRICT_TEMPLATE = """\
[Date]

{company}
[Address]

Dear Hiring Manager,

Hello, my name is {name}, and I submit for your consideration my application for the {position} position.

In my most recent role I have been responsible for [PRIMARY_RESPONSIBILITY_1] and [PRIMARY_RESPONSIBILITY_2].
My role also involves [SECONDARY_RESPONSIBILITY_1], [SECONDARY_RESPONSIBILITY_2], and [SECONDARY_RESPONSIBILITY_3].

{personal_statement}

{company}'s mission to [IMPORTANT_MISSION] is exciting to me. I have been [MISSION_RELATED_WORK],
and [WHY_IMPORTANT]. Supporting {company} in this is something I would love to do.

You can view my work here: {portfolio}. I would appreciate the opportunity to speak further.

Best regards,
{name}"""

    def run(self, cv_text: str, job: Dict[str, str],
            candidate: Dict[str, str],
            mode: str = "free",
            portfolio: str = "",
            personal_statement: str = "") -> Dict[str, Any]:

        if mode == "strict":
            letter = self._STRICT_TEMPLATE.format(
                company=resolve_company_name(job),
                name=candidate.get("name", "Candidate"),
                position=(job.get("title_raw") or "the role"),
                portfolio=portfolio or "[your portfolio URL]",
                personal_statement=personal_statement or "[Add your personal statement here.]",
            )
        else:
            prompt = f"""
You are an expert career storyteller and professional cover letter writer.
Write a compelling, concise cover letter that makes the hiring manager excited to interview this candidate.

Candidate CV:
{cv_text}

Job Posting:
Title: {job.get('title_raw', '')}
URL:   {job.get('url', '')}
Description: {job.get('description', '')}

Candidate Details for Header:
Name:     {candidate.get('name', 'Candidate')}
Email:    {candidate.get('email', '')}
Phone:    {candidate.get('phone', '')}
Location: {candidate.get('location', '')}

Output ONLY the full, final letter text, starting with the header.
            """.strip()
            letter = self.call_openai([AgentMessage("user", prompt)], max_tokens=3000)

        candidate_name = candidate.get("name", "Candidate").strip() or "Candidate"
        # Pass letter text so resolver can fall back to the salutation
        company_name   = resolve_company_name(job, cover_letter_text=letter)
        safe_name    = safe_filename(candidate_name)
        safe_company = safe_filename(company_name)
        pdf_path = str(pathlib.Path(tempfile.gettempdir())
                       / f"{safe_name}_Cover_Letter_{safe_company}.pdf")
        save_cover_letter_pdf(letter, pdf_path)
        return {"letter": letter, "pdf_path": pdf_path}

    def revise(self, original_letter: str, feedback: str,
               cv_text: str, job: Dict[str, str],
               candidate: Dict[str, str]) -> str:
        prompt = f"""
Revise the following cover letter to address the user's feedback while preserving facts.
Improve clarity, specificity, and impact; avoid clichés.

User feedback: {feedback.strip() or '(none provided)'}

CV:
{cv_text}

Original letter:
{original_letter}

Return ONLY the revised full letter starting with the contact header. No commentary.
        """.strip()
        return self.call_openai([AgentMessage("user", prompt)], max_tokens=3000)

### Networking agent

In [11]:
class NetworkingAgent(BaseAgent):

    @staticmethod
    def _clip(s: str, n: int) -> str:
        return (s or "")[:n]

    @staticmethod
    def _safe_json(s: str) -> Dict[str, Any]:
        s = (s or "").strip()
        try:
            return json.loads(s)
        except Exception:
            pass
        m = re.search(r"\{[\s\S]*\}\s*$", s)
        if m:
            try:
                return json.loads(m.group(0))
            except Exception:
                pass
        return {}

    @staticmethod
    def _ensure_subject(email_text: str) -> str:
        t = (email_text or "").strip()
        if not re.match(r"(?i)^subject\s*:", t):
            t = "Subject: Quick question about {{role}} at {{company}}\n\n" + t
        lines   = t.splitlines()
        cleaned = [lines[0]] + [ln for ln in lines[1:]
                                if not re.match(r"(?i)^subject\s*:", ln)]
        return "\n".join(cleaned).strip()

    def _fallback_messages(self, job_url: str = "") -> Dict[str, str]:
        dm = (
            "Hi {{recipient_name}}, I'm exploring the {{role}} role at {{company}} ({{job_link}}). "
            "I've shipped results in similar problem spaces and would value your perspective. "
            "If you have 10-15 minutes this week, could I ask two focused questions about the team's "
            "priorities and what success looks like in the first 90 days? - {{your_name}}"
        )
        email = (
            "Subject: Quick question about {{role}} at {{company}}\n\n"
            "Hi {{recipient_name}},\n\n"
            "I'm preparing to apply for the {{role}} role at {{company}} ({{job_link}}). "
            "My background maps closely to the challenges your team tackles.\n\n"
            "Would you be open to a 10-15 min chat or a quick reply to two specific questions "
            "about priorities and metrics for success? I'll come prepared.\n\n"
            "Thanks,\n{{your_name}}"
        )
        return {"referral_request": dm, "cold_email": email}

    def run(self, cv_text: str, job: Dict[str, str],
            tone: str = "Neutral professional") -> Dict[str, Any]:
        cv    = self._clip(cv_text, 6000)
        jd    = self._clip(job.get("description", "") or "", 4000)
        title = (job.get("title_raw") or "").strip()

        prompt = f"""
You are a principal networking strategist. Generate TWO messages.
Tone: {tone}. Concise, specific, respectful of time.

Constraints:
- LinkedIn DM (\"referral_request\"): 4-10 sentences, no Subject line.
- Cold email (\"cold_email\"): 100-250 words. FIRST LINE must be \"Subject: ...\".
- Use placeholders: {{{{recipient_name}}}}, {{{{your_name}}}}, {{{{role}}}}, {{{{company}}}}, {{{{job_link}}}}.

CV excerpt:   {cv}
Job title:    {title}
Job link:     {job.get('url', '')}
JD excerpt:   {jd}

Return ONE valid JSON object ONLY (no markdown, no commentary):
{{\"referral_request\": \"...\", \"cold_email\": \"Subject: ...\\n\\n...\"}}
        """.strip()

        raw  = self.call_openai([AgentMessage("user", prompt)], max_tokens=3000)
        data = self._safe_json(raw)
        rr   = (data.get("referral_request") or "").strip()
        ce   = (data.get("cold_email") or "").strip()

        if not rr or not ce:
            fb = self._fallback_messages(job.get("url", ""))
            rr = rr or fb["referral_request"]
            ce = ce or fb["cold_email"]

        return {"referral_request": rr, "cold_email": self._ensure_subject(ce)}

    def revise(self, original_msgs: Dict[str, str], feedback: str,
               cv_text: str, job: Dict[str, str],
               candidate_name: str, tone: str = "Neutral professional") -> Dict[str, str]:
        cv = self._clip(cv_text, 6000)
        jd = self._clip(job.get("description", "") or "", 4000)
        fb = (feedback or "Make it sharper and more specific.").strip()

        prompt = f"""
Improve the two messages based on the user's feedback (relationship-first; no direct referral ask).
Tone: {tone}.

User feedback:
{fb}

Original messages (JSON):
{json.dumps(original_msgs, ensure_ascii=False, indent=2)}

CV excerpt:   {cv}
JD excerpt:   {jd}
Candidate:    {candidate_name}

Return ONE valid JSON object ONLY:
{{\"referral_request\": \"...\", \"cold_email\": \"Subject: ...\\n\\n...\"}}
        """.strip()

        raw  = self.call_openai([AgentMessage("user", prompt)], max_tokens=1500)
        data = self._safe_json(raw)
        rr   = (data.get("referral_request") or original_msgs.get("referral_request") or "").strip()
        ce   = (data.get("cold_email")        or original_msgs.get("cold_email")        or "").strip()

        if not rr or not ce:
            fb_msgs = self._fallback_messages(job.get("url", ""))
            rr = rr or fb_msgs["referral_request"]
            ce = ce or fb_msgs["cold_email"]

        return {"referral_request": rr, "cold_email": self._ensure_subject(ce)}

### CV review agent

In [12]:
class CVReviewAgent(BaseAgent):

    def run(self, cv_text: str, job: Dict[str, str]) -> Dict[str, Any]:
        prompt = f"""
You are an executive recruiter whose reputation depends on catching bad candidates before they are hired. Analyse the attached jobseeker's CV against the job description and return STRICT JSON.
Your role is to be objective, analytical, and evidence‑driven at all times. Use the job description to identify key skills, experiences, and qualifications required for the role.
Evaluate the CV's relevance to the job, identifying specific strengths and weaknesses.
Provide a clear verdict on the candidate's fit for the role, supported by evidence from the CV and job description. Avoid generic statements; be specific about which qualifications are met or missing.
Find every flaw, weakness, and area for improvement, no matter how small. Be ruthless in your analysis.

Job Posting:
Title:       {job.get('title_raw', '')}
Description: {job.get('description', '')}

Candidate CV:
{cv_text}

JSON schema:
{{
  \"verdict\": \"Strong Fit - Apply Now\" | \"Good Fit - Minor Revisions Recommended\" | \"Potential Fit - Strategic Repositioning Needed\" | \"Poor Fit - Reconsider\",
  \"overall_confidence\": <float 0.0-1.0>,
  \"summary_analysis\": {{
    \"strengths\": \"...\",
    \"weaknesses\": \"...\",
    \"strategic_angle\": \"...\"
  }},
  \"keyword_optimization\": {{
    \"missing_keywords\": [\"...\"],
    \"overused_keywords\": [\"\"]
  }},
  \"prioritized_edits\": [
    {{
      \"priority\": \"High\" | \"Medium\" | \"Low\",
      \"section\": \"...\",
      \"suggestion\": \"...\",
      \"reasoning\": \"...\",
      \"example_bullets\": [\"...\", \"...\"]
    }}
  ]
}}
        """.strip()

        raw = self.call_openai([AgentMessage("user", prompt)], max_tokens=3000)

        def _extract(text: str) -> Dict[str, Any]:
            text = re.sub(r"```(?:json)?\s*([\s\S]*?)\s*```", r"\1", text, flags=re.I).strip()
            start, depth = None, 0
            for i, ch in enumerate(text):
                if ch == "{":
                    if start is None: start = i
                    depth += 1
                elif ch == "}" and start is not None:
                    depth -= 1
                    if depth == 0:
                        try:
                            return json.loads(text[start:i + 1])
                        except Exception:
                            start = None
            return {}

        try:
            data = json.loads(raw)
        except Exception:
            data = _extract(raw)

        data.setdefault("verdict", "Good Fit - Minor Revisions Recommended")
        data.setdefault("overall_confidence", 0.5)
        data.setdefault("summary_analysis", {})
        data.setdefault("keyword_optimization", {"missing_keywords": [], "overused_keywords": []})
        data.setdefault("prioritized_edits", [])
        return data

### Translation agent

In [13]:
class TranslationAgent(BaseAgent):

    def run(self, text: str, target_lang: str = "Dutch") -> str:
        prompt = f"""
You are a professional translator specialising in career documents.
Translate the following text into {target_lang}.
Maintain professional tone and formatting. No commentary.

Text:
{text}
        """.strip()
        return self.call_openai([AgentMessage("user", prompt)], max_tokens=3000)

### Orchestrator

In [14]:
@dataclass
class Orchestrator:
    cover:      CoverLetterAgent
    net:        NetworkingAgent
    review:     CVReviewAgent
    translator: TranslationAgent

    def route(self, option: str, cv_pdf_path: str,
              job_url: str, jd_text_optional: str = "",
              cover_mode: str = "free", portfolio: str = "",
              personal_statement: str = "") -> Dict[str, Any]:
        cv_text   = extract_text_from_pdf(cv_pdf_path)
        candidate = sniff_contact(cv_text)
        job       = RoleScraper.scrape(job_url)

        raw_desc = (job.get("description") or "").strip()
        needs_jd = len(raw_desc) < 200 and not (jd_text_optional and jd_text_optional.strip())
        if jd_text_optional and len(raw_desc) < 200:
            job["description"] = jd_text_optional

        opt = (option or "").lower().strip()

        if opt == "cover_letter":
            result = self.cover.run(cv_text=cv_text, job=job, candidate=candidate,
                                    mode=cover_mode, portfolio=portfolio,
                                    personal_statement=personal_statement)
            return {
                "type": "cover_letter",
                "cv_text": cv_text, "job": job, "candidate": candidate,
                "letter": result["letter"], "orig_letter": result["letter"],
                "pdf_path": result.get("pdf_path"),
                "messages": None, "orig_messages": None,
                "needs_jd_text": needs_jd,
            }
        elif opt == "networking":
            msgs = self.net.run(cv_text=cv_text, job=job)
            return {
                "type": "networking",
                "cv_text": cv_text, "job": job, "candidate": candidate,
                "letter": None, "orig_letter": None, "pdf_path": None,
                "messages": msgs, "orig_messages": dict(msgs),
                "needs_jd_text": needs_jd,
            }
        elif opt == "cv_review":
            rev = self.review.run(cv_text=cv_text, job=job)
            return {
                "type": "cv_review",
                "cv_text": cv_text, "job": job, "candidate": candidate,
                "review": rev,
                "letter": None, "orig_letter": None, "pdf_path": None,
                "messages": None, "orig_messages": None,
                "needs_jd_text": needs_jd,
            }
        else:
            raise ValueError(f"Unknown option: {option!r}")


### Review coordinator (critic agent)

In [15]:
# Instantiate agents and orchestrator
cover_agent = CoverLetterAgent(
    name="CoverLetterAgent",
    system_prompt="You are an expert cover letter writer for job applications."
)
net_agent = NetworkingAgent(
    name="NetworkingAgent",
    system_prompt="You are an expert networking strategist who helps professionals reach out to industry contacts."
)
review_agent = CVReviewAgent(
    name="CVReviewAgent",
    system_prompt="You are an expert recruiter analyzing CV fit for job roles."
)
translator_agent = TranslationAgent(
    name="TranslationAgent",
    system_prompt="You are a professional translator for career documents."
)

orch = Orchestrator(
    cover=cover_agent,
    net=net_agent,
    review=review_agent,
    translator=translator_agent
)
print("Orchestrator ready ✔️")

Orchestrator ready ✔️


In [16]:
class ReviewCoordinator:
    def __init__(self, cover_agent, net_agent):
        self.cover = cover_agent
        self.net   = net_agent

    def handle(self, state: dict, satisfaction: str, feedback: str) -> dict:
        if not state or not state.get("type"):
            return {**(state or {}), "message": "Run a generator first.", "done": False}

        if satisfaction == "Yes":
            return {**state, "message": "\u2705 Saved.", "done": True}

        fb = (feedback or "").strip() or "Make it clearer and better aligned to the role."

        try:
            if state["type"] == "cover_letter":
                improved = self.cover.revise(
                    original_letter=state.get("letter", ""),
                    feedback=fb,
                    cv_text=state.get("cv_text", ""),
                    job=state.get("job", {}),
                    candidate=state.get("candidate", {}),
                )
                candidate_name = state.get("candidate", {}).get("name", "Candidate").strip() or "Candidate"
                company_name   = resolve_company_name(state.get("job", {}))  # FIX: uses resolver
                safe_name    = re.sub(r"[^A-Za-z0-9]+", "_", candidate_name).strip("_") or "Candidate"
                safe_company = re.sub(r"[^A-Za-z0-9]+", "_", company_name).strip("_")   or "Company"
                pdf_path = str(pathlib.Path(tempfile.gettempdir())
                               / f"{safe_name}_Cover_Letter_{safe_company}.pdf")
                save_cover_letter_pdf(improved, pdf_path)
                return {**state, "letter": improved, "pdf_path": pdf_path,
                        "message": "\U0001f501 Updated per your feedback.", "done": False}

            elif state["type"] == "networking":
                improved = self.net.revise(
                    original_msgs=state.get("messages", {}),
                    feedback=fb,
                    cv_text=state.get("cv_text", ""),
                    job=state.get("job", {}),
                    candidate_name=state.get("candidate", {}).get("name", "Candidate"),
                )
                return {**state, "messages": improved,
                        "message": "\U0001f501 Updated per your feedback.", "done": False}

            else:
                return {**state, "message": "Revision not supported for this type.", "done": False}

        except Exception as e:
            return {**state, "message": f"\u26a0\ufe0f Couldn't apply revision: {e}", "done": False}


REVIEW = ReviewCoordinator(cover_agent=orch.cover, net_agent=orch.net)
print("REVIEW coordinator ready ✔️")

REVIEW coordinator ready ✔️


### Formatters & state helpers

In [17]:
def format_networking(msgs: dict) -> str:
    if not msgs:
        return "_No messages returned._"
    rr = msgs.get("referral_request", "").strip()
    ce = msgs.get("cold_email", "").strip()
    parts = []
    if rr:
        parts.append(f"### Referral request (LinkedIn DM)\n\n{rr}")
    if ce:
        parts.append(f"### Cold email\n\n{ce}")
    return "\n\n---\n\n".join(parts) if parts else "_No messages returned._"


def format_review(r: dict) -> str:
    verdict    = r.get("verdict", "\u2014")
    confidence = r.get("overall_confidence", "\u2014")
    sa         = r.get("summary_analysis", {})
    ko         = r.get("keyword_optimization", {})
    edits      = r.get("prioritized_edits", [])
    md = [
        f"**Verdict:** {verdict}",
        f"**Confidence:** {confidence}",
        "",
        f"**Strengths:** {sa.get('strengths', '\u2014')}",
        f"**Weaknesses:** {sa.get('weaknesses', '\u2014')}",
        f"**Strategic angle:** {sa.get('strategic_angle', '\u2014')}",
    ]
    missing = ko.get("missing_keywords") or []
    if missing:
        md += ["", "**Missing keywords:**", "- " + "\n- ".join(missing)]
    if edits:
        md += ["", "**Prioritised edits:**"]
        for e in edits:
            md += [f"\n**[{e.get('priority','?')}]** *{e.get('section','?')}*: "
                   f"{e.get('suggestion', '')}"]
            md += [f"  \u21b3 _{e.get('reasoning', '')}_"]
            for b in (e.get("example_bullets") or [])[:3]:
                md += [f"  \u2022 {b}"]
    return "\n".join(md)


def clear_state() -> dict:
    return {
        "type": None,
        "cv_text": "", "job": {}, "candidate": {},
        "cv_pdf_path": None,
        "letter": None, "orig_letter": None, "pdf_path": None,
        "messages": None, "orig_messages": None,
    }


def big_notice(title: str, body: str) -> str:
    return (
        f'<div style="border:2px solid #f59e0b;background:#fff7ed;color:#7c2d12;'
        f'border-radius:12px;padding:18px;margin:18px auto;max-width:880px;">'
        f'<div style="font-weight:800;font-size:22px;margin-bottom:6px;">{title}</div>'
        f'<div>{body}</div></div>'
    )

### Gradio flow functions

In [18]:
def run_flow(cv_pdf, job_url, option, jd_text, export_format,
             translate_to_dutch, cover_mode, portfolio, personal_statement, state, progress=gr.Progress()):
    progress(0, desc="Validating inputs...")

    cv_path = None
    if cv_pdf is not None:
        cv_path = cv_pdf.name
    elif state and state.get("cv_pdf_path") and os.path.exists(state["cv_pdf_path"]):
        cv_path = state["cv_pdf_path"]

    if cv_path is None:
        return "Please upload a PDF CV.", None, clear_state(), gr.update(value=None), gr.update(value="")
    if not job_url:
        return "Please paste a job URL.", None, clear_state(), gr.update(value=None), gr.update(value="")

    progress(0.2, desc="Scraping job posting and CV...")
    out = orch.route(option=option, cv_pdf_path=cv_path,
                     job_url=job_url, jd_text_optional=jd_text or "",
                     cover_mode=cover_mode or "free", portfolio=portfolio or "",
                     personal_statement=personal_statement or "")

    if out.get("needs_jd_text"):
        return (
            big_notice("Job page looks gated or empty",
                       "Paste the job description into <b>Optional Job Description text</b> and click <b>Generate</b> again."),
            None, clear_state(), gr.update(value=None), gr.update(value=""),
        )

    progress(0.5, desc="Generating content...")

    if out["type"] == "cover_letter":
        display_text = out["letter"]
    elif out["type"] == "networking":
        display_text = format_networking(out["messages"])
    else:
        display_text = format_review(out["review"])

    if translate_to_dutch:
        progress(0.75, desc="Translating to Dutch...")
        display_text = orch.translator.run(display_text, "Dutch")

    def create_export_file(content: str, file_name: str) -> str:
        ext = export_format.lower()
        safe_name = re.sub(r"[^A-Za-z0-9 _-]+", "_", file_name).strip(" _-") or "output"
        path = str(pathlib.Path(tempfile.gettempdir()) / f"{safe_name}.{ext}")
        if ext == "docx":
            save_text_as_docx(content, path)
        else:
            save_text_as_pdf(content, path)
        return path

    progress(0.9, desc="Creating export file...")

    new_state = {**clear_state(), "cv_pdf_path": cv_path, "type": out["type"],
                 "cv_text": out["cv_text"], "job": out["job"], "candidate": out["candidate"]}

    if out["type"] == "cover_letter":
        candidate_name = out["candidate"].get("name", "Candidate").strip() or "Candidate"
        company_name   = resolve_company_name(out["job"])  # FIX: uses resolver
        safe_name    = re.sub(r"[^A-Za-z0-9]+", "_", candidate_name).strip("_") or "Candidate"
        safe_company = re.sub(r"[^A-Za-z0-9]+", "_", company_name).strip("_")   or "Company"
        generated_file = create_export_file(display_text, f"{safe_name}_Cover_Letter_{safe_company}")
        new_state.update({"letter": out["letter"], "orig_letter": out["letter"],
                          "pdf_path": generated_file})
    elif out["type"] == "networking":
        generated_file = create_export_file(display_text, "Networking_Outreach")
        new_state.update({"messages": out["messages"],
                          "orig_messages": dict(out["messages"])})
    else:
        generated_file = create_export_file(display_text, "CV_Review_Analysis")
        new_state.update({"review": out["review"]})

    progress(1.0, desc="Done!")
    return display_text, generated_file, new_state, gr.update(value=None), gr.update(value="")


def review_step(satisfaction, feedback, export_format, state):
    if not state or not state.get("type"):
        return "Run one of the generators first.", None, state, gr.update(value=None), gr.update(value="")

    updated = REVIEW.handle(state, satisfaction, feedback)

    def _format_and_file(upd):
        t = upd.get("type")
        if t == "cover_letter":
            text = upd.get("letter", "")
            path = upd.get("pdf_path")
        elif t == "networking":
            text = format_networking(upd.get("messages", {}))
            path = str(pathlib.Path(tempfile.gettempdir()) / "networking_messages.md")
            with open(path, "w", encoding="utf-8") as f:
                f.write(text)
        elif t == "cv_review":
            text = format_review(upd.get("review", {}))
            path = str(pathlib.Path(tempfile.gettempdir()) / "cv_review.md")
            with open(path, "w", encoding="utf-8") as f:
                f.write(text)
        else:
            text, path = upd.get("message", ""), None
        return text, path

    text, path = _format_and_file(updated)
    return text, path, updated, gr.update(value=None), gr.update(value="")

### Gradio UI

In [19]:
with gr.Blocks(title="Job Hunt Agents") as demo:
    gr.Markdown("## Job Hunt Agents\nUpload your CV (PDF) + job link \u2192 choose what to generate.")

    with gr.Row():
        cv_pdf  = gr.File(label="CV (PDF)", file_types=[".pdf"])
        job_url = gr.Textbox(label="Job link", placeholder="https://...")

    jd_text = gr.Textbox(
        label="Optional Job Description text (if the link is gated/login-only)",
        lines=5,
        placeholder="Paste the job description here if the page is behind a login.",
    )

    with gr.Row():
        option = gr.Radio(
            choices=[("Cover letter", "cover_letter"),
                     ("Networking",   "networking"),
                     ("CV review",    "cv_review")],
            value="cover_letter", label="1. Choose Agent",
        )
        export_format = gr.Radio(
            choices=["PDF", "DOCX"], value="PDF", label="2. Export format",
        )

    with gr.Row():
        translate_to_dutch = gr.Checkbox(label="Translate output to Dutch", value=False)
        cover_mode = gr.Radio(
            choices=[("Free Style (AI-written)", "free"),
                     ("Strict Formula (template)", "strict")],
            value="free", label="Cover letter mode",
        )

    with gr.Accordion("Cover letter extras (optional)", open=False):
        portfolio = gr.Textbox(label="Portfolio URL",
                               placeholder="https://yourportfolio.com")
        personal_statement = gr.Textbox(
            label="Personal statement (used in Strict Formula mode)",
            lines=3,
            placeholder="A short paragraph about your motivation or values...",
        )

    run_btn = gr.Button("Generate", variant="primary")

    output_md = gr.Markdown(label="Result")
    file_out  = gr.File(label="Download", interactive=False)

    gr.Markdown("### Review the result")
    satisfaction = gr.Radio(choices=["Yes", "No"], value=None,
                            label="Are you satisfied with the output?")
    feedback = gr.Textbox(label="If 'No', what should we change?", lines=4,
                          placeholder="e.g., shorter intro, stronger metrics, warmer tone")
    apply_btn = gr.Button("Apply review / Improve")

    state = gr.State(clear_state())

    run_btn.click(
        fn=run_flow,
        inputs=[cv_pdf, job_url, option, jd_text, export_format,
                translate_to_dutch, cover_mode, portfolio, personal_statement, state],
        outputs=[output_md, file_out, state, satisfaction, feedback],
    )
    apply_btn.click(
        fn=review_step,
        inputs=[satisfaction, feedback, export_format, state],
        outputs=[output_md, file_out, state, satisfaction, feedback],
    )

### Launch

In [ ]:
demo.queue().launch(share=True, debug=True)

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://590cfc283f0c2ec5ee.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/gradio/queueing.py", line 867, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<5 lines>...
    )
    ^
  File "/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/gradio/route_utils.py", line 374, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<11 lines>...
    )
    ^
  File "/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/gradio/blocks.py", line 2179, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<8 lines>...
    )
    ^
  File "/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/gradio/blocks.py", line 1636, in call_function
    prediction = await anyio.to_thread.run_